# 03 — Estimating Intrinsic Parameters (K, D)

**Objective:** Run `cv2.calibrateCamera` on the valid corner detections from Notebook 02, inspect the estimated K and D, save a calibration profile, and understand what the numbers mean.

**What this notebook produces:**
- Intrinsic matrix **K** (focal lengths + principal point)
- Distortion vector **D** (k₁ k₂ p₁ p₂ k₃)
- RMS reprojection error
- A saved `configs/saved_profiles/` YAML file reusable in any later notebook


In [ ]:
import sys, logging
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt

SRC = Path('..') / 'src'
if str(SRC.resolve()) not in sys.path:
    sys.path.insert(0, str(SRC.resolve()))

from calibvision.calibrator import CameraCalibrator
from calibvision.profile import CalibrationProfile
from calibvision.utils import load_images_from_dir, draw_reprojection_error

logging.basicConfig(level=logging.INFO, format='%(levelname)s  %(message)s')

PROJECT_ROOT = Path('..').resolve()
CALIB_DIR    = PROJECT_ROOT / 'data' / 'calibration_images'
PROFILE_DIR  = PROJECT_ROOT / 'configs' / 'saved_profiles'
PROFILE_DIR.mkdir(parents=True, exist_ok=True)

BOARD_COLS, BOARD_ROWS = 9, 6
print('Setup complete')

In [ ]:
# ── 3.1  Re-run corner detection (idempotent) ─────────────────────────────────
paths, _ = load_images_from_dir(CALIB_DIR)
calibrator = CameraCalibrator(board_cols=BOARD_COLS, board_rows=BOARD_ROWS,
                               min_valid_frames=10)
detections = calibrator.detect_corners(paths, verbose=False)
n_ok = sum(d.success for d in detections)
print(f'Valid detections: {n_ok} / {len(detections)}')

In [ ]:
# ── 3.2  Calibrate ────────────────────────────────────────────────────────────
result = calibrator.calibrate(detections)

K = result.camera_matrix
D = result.dist_coeffs
rms = result.rms_reprojection_error

print(f'\n=== Calibration Results ===')
print(f'RMS reprojection error: {rms:.4f} px')
if rms < 0.5:
    print('  Quality: Excellent (< 0.5 px)')
elif rms < 1.0:
    print('  Quality: Acceptable (< 1.0 px)')
else:
    print('  Quality: Poor (> 1.0 px) — review image set')

print(f'\nIntrinsic matrix K:')
print(K)
print(f'\nDistortion coefficients D = [k1, k2, p1, p2, k3]:')
print(D.ravel())

In [ ]:
# ── 3.3  Interpret the intrinsic values ──────────────────────────────────────
fx, fy = K[0, 0], K[1, 1]
cx, cy = K[0, 2], K[1, 2]
w, h   = result.image_size
fov_x  = 2 * np.degrees(np.arctan(w / (2 * fx)))
fov_y  = 2 * np.degrees(np.arctan(h / (2 * fy)))
aspect  = fx / fy   # should be ≈1.0 for square pixels

print('=== Intrinsic Interpretation ===')
print(f'Image size:       {w}×{h} px')
print(f'Focal length:     fx={fx:.2f} px   fy={fy:.2f} px')
print(f'Pixel aspect:     {aspect:.4f}  (1.0 = square pixels, expected for modern sensors)')
print(f'Principal point:  cx={cx:.2f}   cy={cy:.2f}')
print(f'Expected centre:  {w/2:.1f}   {h/2:.1f}')
print(f'Horizontal FOV:   {fov_x:.1f}°')
print(f'Vertical FOV:     {fov_y:.1f}°')

D_flat = D.ravel()
print(f'\n=== Distortion Interpretation ===')
print(f'k1={D_flat[0]:.6f}  (radial 2nd order: negative=barrel, positive=pincushion)')
print(f'k2={D_flat[1]:.6f}  (radial 4th order)')
print(f'p1={D_flat[2]:.6f}  (tangential)')
print(f'p2={D_flat[3]:.6f}  (tangential)')
print(f'k3={D_flat[4]:.6f}  (radial 6th order)')
if abs(D_flat[0]) < 0.01:
    print('  → Very low distortion (expected for synthetic images)')

In [ ]:
# ── 3.4  Per-view reprojection error chart ────────────────────────────────────
names = [p.name for p in result.valid_image_paths]
fig = draw_reprojection_error(
    result.per_view_errors, names, rms, max_acceptable=1.0
)
plt.show()

# Flag high-error images
high_error = [(n, e) for n, e in zip(names, result.per_view_errors) if e > 1.0]
if high_error:
    print('\n⚠️  High-error images (consider removing):')
    for name, err in sorted(high_error, key=lambda x: -x[1]):
        print(f'   {name}: {err:.4f} px')
else:
    print('\n✓ All per-view errors within acceptable range')

In [ ]:
# ── 3.5  Save calibration profile ────────────────────────────────────────────
profile = calibrator.to_profile(
    result,
    camera_name='synthetic_9x6_demo',
    notes=f'Generated from {n_ok} synthetic images. Not a real camera profile.'
)

profile_path = PROFILE_DIR / 'synthetic_demo.yaml'
profile.save(profile_path)

print(profile.summary())

# Verify round-trip: reload and compare
loaded = CalibrationProfile.load(profile_path)
K_diff = np.max(np.abs(loaded.camera_matrix - profile.camera_matrix))
D_diff = np.max(np.abs(loaded.dist_coeffs - profile.dist_coeffs))
print(f'\nRound-trip check — max K diff: {K_diff:.2e}  max D diff: {D_diff:.2e}')
assert K_diff < 1e-9, 'K round-trip mismatch!'
assert D_diff < 1e-9, 'D round-trip mismatch!'
print('✓ Profile save/load round-trip verified')

## Summary

| Output | Value |
|---|---|
| RMS reprojection error | *(see Cell 3.2)* |
| Horizontal FOV | *(see Cell 3.3)* |
| Profile saved to | `configs/saved_profiles/synthetic_demo.yaml` |
| Round-trip verified | ✓ |

## Limitation

This profile was estimated from **synthetic** images without real lens distortion. The k1–k3 values will be near-zero. To calibrate a real camera, capture 15–30 photos of a printed checkerboard from diverse angles.

## Next step

Open `04_lens_distortion_correction.ipynb` to apply the profile to new images.
